In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys
import requests
import pandas as pd
import time

# --- 1. Environment & Central Schema Setup ---
API_BASE = "https://thesauri.cessda.eu/rest/v1/elsst-6"

notebook_dir = os.path.dirname(os.path.abspath("__file__"))
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

# Import Schema Manager
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup (Auto-Stubbing Enabled) ---
SOURCE_PREFIX = "ELSST"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="European Language Social Science Thesaurus",
    fallback_uri="https://elsst.cessda.eu/id/6/"
)
SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']

raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")

# Verified Seed List
TOP_LEVEL_URIS = [
    "https://elsst.cessda.eu/id/6/28ce4564-fb7b-48a3-81ff-2e31dcfb508b", # BELIEFS
    "https://elsst.cessda.eu/id/6/0a7efa6d-2063-4ad2-b3cc-13e2306c2dc1", # CHURCH AND STATE
    "https://elsst.cessda.eu/id/6/b4067811-ede2-4338-a1bf-b751800ae394", # ESOTERIC PRACTICES
    "https://elsst.cessda.eu/id/6/a80484f0-51cd-4730-b544-1465510f7f8e", # ETHICS
    "https://elsst.cessda.eu/id/6/575d20f3-b0f9-4794-b776-1e120c557d00", # PHILOSOPHY
    "https://elsst.cessda.eu/id/6/6cf5a38e-d0cc-4b06-9917-659c66adadf1", # RELIGION
    "https://elsst.cessda.eu/id/6/ce795285-4d4d-496d-83be-d79d77c5baf9", # ANTI-SEMITISM
    "https://elsst.cessda.eu/id/6/444b37b0-9004-4975-8d02-d2651964c6ed", # RELIGIOUS LEADERS
    "https://elsst.cessda.eu/id/6/b8ae6bd1-45a1-43d4-a0b2-d753ed94d143", # FAITH SCHOOLS
    "https://elsst.cessda.eu/id/6/2bf02f5d-d189-4651-82ae-4cb9fd375b9a", # JEWS
    "https://elsst.cessda.eu/id/6/a63185a1-cc79-4c0c-abca-ed6cd0e40743", # RELIGION AND THEOLOGY EDUCATION
    "https://elsst.cessda.eu/id/6/49384517-5a79-4cae-bd92-3dbdb1d06431", # RELIGIOUS BUILDINGS
    "https://elsst.cessda.eu/id/6/21e90540-27f2-40c3-81e1-c045a06fb9ba", # RELIGIOUS DISCRIMINATION
    "https://elsst.cessda.eu/id/6/0519458c-418d-4a4f-810b-5c78cacf4c4a", # RELIGIOUS FREEDOM
    "https://elsst.cessda.eu/id/6/730df5ec-cb2e-4086-8f5d-f2c429fc4353", # RELIGIOUS INSTRUCTION
    "https://elsst.cessda.eu/id/6/583acf15-cabc-45ab-a4a8-0912814a01ea", # RELIGIOUS SOCIETIES
    "https://elsst.cessda.eu/id/6/17b4a01c-6277-46c3-88d5-dd47370d7dc2"  # SPIRITUAL HEALING
]

HEADERS = {'Accept': 'application/json'}
processed_ids = set()

# --- 3. Helper Functions ---
def get_english_value(data_item):
    """Extracts English prefLabel or definition from ELSST's multilingual arrays."""
    if isinstance(data_item, list):
        for entry in data_item:
            if isinstance(entry, dict) and entry.get('lang') == 'en':
                return entry.get('value', '')
    elif isinstance(data_item, dict):
        return data_item.get('value', '')
    return str(data_item) if data_item else ""

def has_non_english(data_item):
    """Detects if any non-English tags exist in the ELSST array."""
    if isinstance(data_item, list):
        for entry in data_item:
            if isinstance(entry, dict) and entry.get('lang', 'en') != 'en':
                return True
    return False

def get_parent_info(uri):
    """Looks 'up' the tree once to find the parent for a specific seed."""
    url = f"{API_BASE}/data?uri={uri}&lang=en"
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code == 200:
            graph = res.json().get('graph', [])
            node = next((n for n in graph if n.get('uri') == uri), {})
            broader = node.get('broader')
            if broader:
                b_uri = broader[0].get('uri') if isinstance(broader, list) else broader.get('uri')
                if b_uri:
                    b_label_url = f"{API_BASE}/label?uri={b_uri}&lang=en"
                    b_res = requests.get(b_label_url, headers=HEADERS, timeout=10)
                    b_label = b_res.json().get('prefLabel', 'Unknown Parent')
                    return b_uri.split('/')[-1], b_label
    except: pass
    return "", ""

def get_concept_details(uri):
    """Fetches full details for a concept node."""
    url = f"{API_BASE}/data?uri={uri}&lang=en"
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code == 200:
            graph = res.json().get('graph', [])
            node = next((n for n in graph if n.get('uri') == uri), {})
            
            label = get_english_value(node.get('prefLabel', ''))
            
            alts_raw = node.get('altLabel', [])
            if not isinstance(alts_raw, list): alts_raw = [alts_raw]
            synonyms = [a.get('value') for a in alts_raw if isinstance(a, dict) and a.get('lang') == 'en']
            
            defn = get_english_value(node.get('definition', ''))
            if not defn: defn = get_english_value(node.get('skos:scopeNote', ''))
                
            # Detect translations
            has_trans = "yes" if has_non_english(node.get('prefLabel', [])) or has_non_english(alts_raw) else ""
                
            return {
                'label': label, 
                'synonyms': " | ".join(synonyms), 
                'description': defn, 
                'has_translation': has_trans
            }
    except: pass
    return None

def ingest_recursive(uri, path="", p_id=""):
    """Crawls down the 'narrower' (child) branches of the tree."""
    cid = uri.rstrip('/').split('/')[-1]
    if cid in processed_ids: return
    
    details = get_concept_details(uri)
    if not details or not details['label']: return
    
    current_label = details['label']
    current_path = f"{path} > {current_label}" if path else current_label
    
    display_text = f"Ingesting: {current_path[:90]}..."
    print(f"\r{display_text}{' ' * 60}", end="", flush=True)
    
    # Pack the raw data
    extracted_data = {
        'Source_System': SOURCE_NAME, 
        'Primary_Label': current_label,
        'CURIE': f"{SOURCE_PREFIX}:{cid}", 
        'Formal_Label': current_label,
        'Concept_Type': 'skos:Concept',
        'Hierarchy_Path': current_path, 
        'Synonyms': details['synonyms'],
        'Description': details['description'], 
        'Parent_IDs': p_id,
        'Concept_ID': cid, 
        'URI': uri, 
        'Has_Translation': details['has_translation'],
        'Status': 'active'
    }
    
    # MAGIC HAPPENS HERE: Force data into the strict 14-column schema
    clean_row = finalize_row(extracted_data)
    
    df_row = pd.DataFrame([clean_row])[COLUMN_ORDER]
    file_exists = os.path.isfile(raw_ingest_file)
    df_row.to_csv(raw_ingest_file, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
    
    processed_ids.add(cid)

    # Recurse into children
    narrower_url = f"{API_BASE}/narrower?uri={uri}&lang=en"
    try:
        n_res = requests.get(narrower_url, headers=HEADERS, timeout=10)
        if n_res.status_code == 200:
            for child in n_res.json().get('narrower', []):
                child_uri = child.get('uri')
                if child_uri:
                    time.sleep(0.1)
                    ingest_recursive(child_uri, current_path, cid)
    except: pass

# --- 4. Execution ---
if os.path.exists(raw_ingest_file): os.remove(raw_ingest_file)

print(f"Starting {SOURCE_PREFIX} Hierarchy-Aware Ingest...")

try:
    for uri in TOP_LEVEL_URIS:
        parent_id, parent_label = get_parent_info(uri)
        ingest_recursive(uri, path=parent_label, p_id=parent_id)
finally:
    print(f"\n\nIngest Complete! {len(processed_ids)} records saved to: {os.path.basename(raw_ingest_file)}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Starting ELSST Hierarchy-Aware Ingest...
Ingesting: COMPLEMENTARY THERAPIES > SPIRITUAL HEALING...                                                                                                      

Ingest Complete! 111 records saved to: raw_elsst.csv
